# Plot Zebra + Co-click From Real Data

Load a saved memory pickle and generate zebra GIF/heatmaps.

In [9]:
import pickle
from pathlib import Path
import numpy as np

from qiskit_quantuminspire.qi_provider import QIProvider
from tuna_fid_single_job import save_memory_3d_plots


In [10]:
mem_path = Path("data") / "fid_job_memory_noreset_large_1404.pkl"
with mem_path.open("rb") as f:
    loaded = pickle.load(f)

if isinstance(loaded, dict) and "memory" in loaded:
    mem = loaded["memory"]
    taus = loaded.get("taus")
    shots = loaded.get("shots")
    echo = loaded.get("echo")
    reset = loaded.get("reset")
else:
    # Backward compatibility with old pickles that contain only memory.
    mem = loaded
    taus = None
    shots = None
    echo = True
    reset = False

if taus is None:
    raise ValueError("No taus found in pickle. Re-run acquisition notebook to save metadata.")

taus = [float(t) for t in taus]
n_tau = len(taus)

provider = QIProvider()
backend = provider.get_backend("Tuna-17")
num_qubits = backend.num_qubits

print(f"Loaded {len(mem)} shots, qubits={num_qubits}, n_tau={n_tau}")
print(f"Metadata: shots={shots}, echo={echo}, reset={reset}")

Loaded 5000 shots, qubits=17, n_tau=50
Metadata: shots=5000, echo=False, reset=False


In [5]:
out_dir = Path("figures")
out_dir.mkdir(parents=True, exist_ok=True)
prefix = "fid_memory_3d"

tau_arr = np.asarray(taus, dtype=float)

# 1) Shot-by-shot GIF for compact output: first 1000 shots, 15 fps, dpi 80.


# 2) 100-shot averaged GIF sped up to 5 fps.
p_tilted, p_gif_3d, p_gif_2d_bin100, p_gif_2d_per_shot, p_coclick, p_coclick_excess, p_tau_rep, p_derived_npz = save_memory_3d_plots(
    mem,
    num_qubits,
    n_tau,
    out_dir,
    max_shots=shots or len(mem),
    prefix=prefix,
    tau_ns=tau_arr,
    gif_2d_fps=4,
    gif_2d_dpi=100,
    gif_2d_marginal_history=10,
    gif_2d_rep_bin=100,
    reset_qubits=bool(reset),
)

max_shots_per_shot = min(1000, shots or len(mem))
_, _, p_gif_2d_per_shot_fast, _, _, _, _, _ = save_memory_3d_plots(
    mem,
    num_qubits,
    n_tau,
    out_dir,
    max_shots=max_shots_per_shot,
    prefix=f"{prefix}_first{max_shots_per_shot}_shotbyshot",
    tau_ns=tau_arr,
    gif_2d_fps=15,
    gif_2d_dpi=80,
    gif_2d_marginal_history=10,
    gif_2d_rep_bin=1,
    reset_qubits=bool(reset),
)

print("Wrote shot-by-shot GIF:", p_gif_2d_per_shot_fast)
print("Wrote 100-shot-avg GIF:", p_gif_2d_bin100)
print("Wrote:", p_coclick)
print("Wrote:", p_coclick_excess)
print("Wrote:", p_tau_rep)
print("Wrote:", p_derived_npz)

Wrote shot-by-shot GIF: figures/fid_memory_3d_first1000_shotbyshot_repetitions_2d.gif
Wrote 100-shot-avg GIF: figures/fid_memory_3d_repetitions_2d_bin100.gif
Wrote: figures/fid_memory_3d_qubit_coclick.png
Wrote: figures/fid_memory_3d_qubit_coclick_excess.png
Wrote: figures/fid_memory_3d_tau_vs_repetition_meanq.png
Wrote: figures/fid_memory_3d_memory_derived.npz


In [14]:
from device_benchmark_scatter import plot_device_benchmark_planes

bench_2d_path = out_dir / f"{prefix}_device_benchmark_planes.png"
p_bench_2d = plot_device_benchmark_planes(bench_2d_path)
print("Wrote device benchmark 2D planes:", p_bench_2d)

Wrote device benchmark 2D planes: figures/fid_memory_3d_device_benchmark_planes.png


In [11]:
from device_benchmark_bars import plot_device_benchmark_bars

bench_path = out_dir / f"{prefix}_device_benchmarks.png"
p_bench = plot_device_benchmark_bars(
    out_path=bench_path,
    backend=backend,
    device_label=getattr(backend, "name", "Tuna-17"),
    use_reference_fallback=True,
)
print("Wrote device benchmarks:", p_bench)

AttributeError: 'QIBackend' object has no attribute 'properties'

In [ ]:
backend.

<qiskit_quantuminspire.qi_backend.QIBackend object at 0x1334ad760 (name=Tuna-17, id=7)>